# Simple PDF Analysis with VLM

Minimal example showing how to analyze a PDF using RC GenAI's VLM features. 


## How This Works

### What is a Vision-Language Model (VLM)?
A VLM is an AI that can both "see" images AND understand text.

**Traditional text-only LLMs:**
- Can only read extracted text
- Miss figures, equations, handwritten notes
- Need optical character recognition (OCR) preprocessing

**Vision-Language Models:**
- Analyze PDFs as screenshots/images
- See figures, diagrams, tables, and text
- Handle complex layouts automatically

### The Process:
1. **PDF** -> Convert page to image (PNG)
2. **Image** -> Encode as base64 for API
3. **API** -> VLM "looks" at the image
4. **Response** -> Answers your question about what it sees

**Perfect for:**
- Research papers with figures
- Handwritten documents
- Forms and scanned PDFs
- Documents where layout matters

## Step 1: Install Required Packages

This checks if `requests` and `pymupdf` are installed, and installs them if needed.

In [ ]:
# Check and install required packages
import subprocess
import sys

required_packages = {
    'requests': 'requests',
    'pymupdf': 'pymupdf',  
}

missing_packages = []

for package, pip_name in required_packages.items():
    try:
        __import__(package)
        print(f' {package} is already installed')
    except ImportError:
        missing_packages.append(pip_name)
        print(f'⚠️  {package} not found, will install {pip_name}...')

if missing_packages:
    print(f'\n📦 Installing missing packages: {missing_packages}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing_packages)
    print('\n Installation complete!')
else:
    print('\n All required packages are available')

## Step 2: Import Libraries

⚠️ Set `UVARC_GenAI_API` environment variable in `~/.bashrc` before running this notebook.

In [ ]:
import os
import requests
import pymupdf
from IPython.display import Markdown, display

# Configuration
API_URL = 'https://open-webui.rc.virginia.edu/api/chat/completions'
API_KEY = os.getenv('UVARC_GenAI_API')
MODEL = '<model>' #insert custom model name with streaming disabled

# Check if API key is set
if not API_KEY:
    print('⚠️  WARNING: UVA_GenAI_API environment variable not set!')
    print('   Export your API key before running: export UVA_GenAI_API=your_key_here')
else:
    print('✅ API key found')

## Step 3: Get a PDF

We'll download a sample research paper from arXiv. You can replace this URL with your own PDF.

In [ ]:
# Download sample PDF
resp = requests.get('https://arxiv.org/pdf/1706.03762.pdf', timeout=30)
open('paper.pdf', 'wb').write(resp.content)
print('PDF downloaded: paper.pdf')

## Step 4: Convert PDF to Images

VLMs need images so we convert each page of the PDF to a PNG image.

💡 **Why convert?** VLMs "see" documents like humans do - by looking at the visual layout, not reading raw text.

In [ ]:
# Extract all pages as images (or use max_pages=N for first N pages)
def extract_images(pdf_path, max_pages=None):
    doc = pymupdf.open(pdf_path)
    images = []
    pages_to_export = doc if max_pages is None else doc[:max_pages]
    for page in pages_to_export:
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))  # 2x zoom for better quality
        images.append(pix.tobytes('png'))
    doc.close()
    return images

# Gets first three pages (change max_pages=None to extract all pages)
images = extract_images('paper.pdf', max_pages=3)
print(f' Extracted {len(images)} pages as images')


## Step 5: Set Up VLM Function

In [ ]:
import base64

def ask_vlm(image_bytes, question):
    headers = {
        'Authorization': f'Bearer {API_KEY}', 
        'Content-Type': 'application/json'
    }
    
    image_b64 = base64.b64encode(image_bytes).decode()
    
    payload = {
        'model': MODEL,
        'messages': [{
            'role': 'user',
            'content': [
                {'type': 'text', 'text': question},
                {'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{image_b64}'}}
            ]
        }],
        'stream': False
    }
    
    resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
    return resp.json()['choices'][0]['message']['content']

def show_md(response):
    display(Markdown(response))

print(' VLM functions ready')

## Step 6: Analyze the PDF

Now ask the VLM anything about the document!

**Example questions you can ask:**
- What is the main contribution of this paper?
- List all authors and affiliations
- Are there any figures showing [specific topic]?
- Extract the methodology section
- Summarize the abstract in 2 sentences
- What equations are shown on this page?

In [ ]:
# Analyze first page
print(f' Analyzing page 1 of {len(images)}...')
answer = ask_vlm(images[0], "What is this paper about? Who wrote it?")
show_md(answer)

## Try Different Questions

In [ ]:
# Extract specific details
show_md(ask_vlm(images[0], "List all authors with their affiliations in bullet points"))

In [ ]:
# Analyze visual content  (page 3)
show_md(ask_vlm(images[2], "Are there any figures, tables, or equations on this page? Describe them in detail."))

In [ ]:
# Get structured output (JSON)
show_md(ask_vlm(images[0], "Give me JSON format: title, authors (array), publication year, main contribution (1 sentence)"))

## 🔄 Try Your Own PDF

Adapt this notebook for your own documents:

### Option 1: Local file
```python
# Replace Step 3 with:
pdf_path = "/path/to/your/paper.pdf"
images = extract_images(pdf_path, max_pages=3)  # Analyze first 3 pages only
```

### Option 2: Different URL
```python
# Replace Step 3 with:
resp = requests.get('https://your-domain.com/paper.pdf')
open('my_paper.pdf', 'wb').write(resp.content)
images = extract_images('my_paper.pdf', max_pages=5)
```

### Option 3: Multiple pages at once
```python
# Analyze all pages and combine results
for i, img in enumerate(images[:3]):  # First 3 pages
    print(f"\n--- Page {i+1} ---")
    show_md(ask_vlm(img, "Summarize this page in one sentence"))
```

**Tips:**
- Start with `max_pages=1` or `2` to test (cheaper, faster)
- Ensure PDF has selectable text (scanned images work but cost more)
- Clear, high-resolution PDFs give best results

In [ ]:
# Ask VLM to create BibTeX citation
bib_prompt = '''Create a BibTeX citation for this paper.

Requirements:
- Use @article format
- Include all authors with proper BibTeX formatting (and between multiple authors)
- Include title, venue (full conference name), year
- Add arXiv ID if visible

Return ONLY the BibTeX entry, no markdown formatting.'''

print('📝 Generating BibTeX citation...')
bibtex = ask_vlm(images[0], bib_prompt)

print("\nGenerated Citation:")
print("="*60)
print(bibtex)
print("="*60)

In [ ]:
# Save citation to .bib file
bib_filename = 'paper_citation.bib'
with open(bib_filename, 'w') as f:
    f.write(bibtex)
print(f'✅ Saved citation to: {bib_filename}')
print(f'\nYou can now use this file with LaTeX or reference managers like Zotero/Mendeley')

##  Done!

### The VLM handled:
- Text recognition 
- Layout understanding
- Figure/table analysis
- Citation formatting
- Following your instructions


---

### 💡 Next Steps:
1. Try analyzing other documents
2. Experiment with different questions (figure descriptions, methodology extraction, etc.)
3. Combine multiple pages for full document summaries

### ⚠️ Important Notes:
- **API Key:** Keep your `UVARC_GenAI_API` key private; do not share notebooks with keys
- **Privacy:** Do not upload highly-sensitive/confidential documents to API

This example works with any PDF URL or local file. Just replace the download step!